In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNetCV, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import LinearSVR
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error
from scipy.stats import pearsonr
from sklearn.linear_model import TweedieRegressor, BayesianRidge, HuberRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import ARDRegression

In [2]:
MODEL_TYPE = 'elasticnet'
USE_HORVATH = False
FEATURE_K = 100000

plot_dir = f"blood/plots/{MODEL_TYPE}"
os.makedirs(plot_dir, exist_ok=True)

In [3]:
metadata = pd.read_csv('meta_bhak.csv', sep=';', index_col=0)

In [4]:
metadata

,Age (years),Gender,Condition
acc,,,
SRR9190548,36,Female,MDD
SRR9190515,27,Male,Healthy
SRR9190731,44,Female,SA
SRR9190772,51,Male,MDD
SRR9190614,23,Female,Healthy
...,...,...,...
SRR9190717,26,Female,Healthy
SRR9190489,40,Female,SA
SRR9190576,32,Male,SA


In [5]:
df = pd.read_csv('train_multi_omics_bhak.csv', sep=';', index_col=0)

In [6]:
X = df.T

In [7]:
samples = X.index.intersection(metadata.index)

In [8]:
X = X.loc[samples]
y = metadata.loc[samples, 'Age (years)']

In [9]:
X = X.astype('float32')

In [10]:
def horvath_transform(age, adult_age=20):
    if not USE_HORVATH: return age
    age = np.array(age)
    return np.where(age <= adult_age, np.log(age+1)-np.log(adult_age + 1), (age - adult_age)/(adult_age+1))

In [11]:
def horvath_inverse(transformed_age, adult_age=20):
    if not USE_HORVATH: return transformed_age
    transformed_age = np.array(transformed_age)
    return np.where(transformed_age < 0, np.exp(transformed_age + np.log(adult_age+1))-1, 
                    transformed_age * (adult_age+1) + adult_age)

In [12]:
y_transformed = horvath_transform(y)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [14]:
from sklearn.feature_selection import SelectKBest, f_regression

def perform_feature_selection(X, y, k=50000):
    print(f"Selecting top {FEATURE_K} features...")
    selector = SelectKBest(score_func=f_regression, k=FEATURE_K)
    X_selected = selector.fit_transform(X, y)
    
    selected_features = X.columns[selector.get_support()]
    
    print(f"Final feature count: {X_selected.shape[1]}")
    return X_selected, selected_features

X_reduced, clock_sites = perform_feature_selection(X_train, y_train, k=150000)

Selecting top 100000 features...
Final feature count: 100000


In [15]:
X_train_reduced = X_reduced
X_test_reduced = X_test[clock_sites]

In [16]:
models = {
    'elasticnet': ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 1.0], cv=5, n_jobs=-1, max_iter=10000),
    'ridge': RidgeCV(cv=5),
    'lasso': LassoCV(cv=5, n_jobs=-1),
    'randomforest': RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42),
    'extratrees': ExtraTreesRegressor(n_estimators=100, n_jobs=-1, random_state=42),
    'histgradient': HistGradientBoostingRegressor(max_iter=500, random_state=42),
    'pls': PLSRegression(n_components=20),
    'linearsvr': LinearSVR(max_iter=10000, random_state=42),
    'bayesian_ridge': BayesianRidge(max_iter=1000),
    'huber': make_pipeline(StandardScaler(), HuberRegressor(max_iter=1000)),
    'tweedie': make_pipeline(StandardScaler(), TweedieRegressor(power=1.5, link='log', max_iter=1000)),
    'ard': ARDRegression(),
}

In [ ]:
model = models[MODEL_TYPE]
print(f"Training {MODEL_TYPE}...")
model.fit(X_train_reduced, y_train)

Training elasticnet...


In [ ]:
preds_transformed = model.predict(X_test_reduced)
if MODEL_TYPE == 'pls': preds_transformed = preds_transformed.flatten()
preds_years = horvath_inverse(preds_transformed)
actual_years = horvath_inverse(y_test)

In [ ]:
metrics = {
    "MAE": mean_absolute_error(actual_years, preds_years),
    "MedianAE": median_absolute_error(actual_years, preds_years),
    "R2": r2_score(actual_years, preds_years),
    "Pearson_R": pearsonr(actual_years, preds_years)[0]
}

In [ ]:
print("Final Metrics")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(actual_years, preds_years, alpha=0.6, color='teal', edgecolors='white')
lims = [0, 100]
plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0)
plt.xlabel("Actual Age (Years)")
plt.ylabel("Predicted Age (Years)")
plt.title(f"Model: {MODEL_TYPE} | Horvath: {USE_HORVATH}\nMAE: {metrics['MAE']:.2f} | R2: {metrics['R2']:.2f}\nMedianAE: {metrics['MedianAE']:.2f} | Pearson_R: {metrics['Pearson_R']:.2f}")

filename = f"{MODEL_TYPE}_h{USE_HORVATH}_k{FEATURE_K}.png"
plt.savefig(os.path.join(plot_dir, filename))
print(f"\nPlot saved to: {os.path.join(plot_dir, filename)}")
plt.show()

In [ ]:
residuals = actual_years - preds_years
plt.figure(figsize=(8, 4))
plt.scatter(actual_years, residuals, alpha=0.5, color='coral')
plt.axhline(y=0, color='black', linestyle='--')
plt.title(f"Residual Plot: {MODEL_TYPE}")
plt.xlabel("Actual Age")
plt.ylabel("Error (Years)")
filename = f"{MODEL_TYPE}_h{USE_HORVATH}_k{FEATURE_K}_residuals.png"
plt.savefig(os.path.join(plot_dir, filename))

In [ ]:
%pip install seaborn

In [ ]:
import seaborn as sns
eaa = preds_years - actual_years
sns.histplot(eaa, kde=True, color='skyblue')
plt.axvline(x=0, color='red', linestyle='--')
plt.title("Distribution of Epigenetic Age Acceleration")
plt.xlabel("Years of Acceleration (Predicted - Actual)")
filename = f"{MODEL_TYPE}_h{USE_HORVATH}_k{FEATURE_K}_eaa.png"
plt.savefig(os.path.join(plot_dir, filename))

In [ ]:
if hasattr(model, 'coef_') or hasattr(model, 'feature_importances_'):
    # 1. Extract and force to 1D (flatten)
    if hasattr(model, 'coef_'):
        importances = np.array(model.coef_).flatten()
    else:
        importances = np.array(model.feature_importances_).flatten()
    abs_weights = np.abs(importances)
    indices = np.argsort(abs_weights)[-20:]
    top_weights = importances[indices]
    top_names = [clock_sites[i] for i in indices]

    # 4. Plot
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), top_weights, color='plum')
    plt.yticks(range(len(indices)), top_names)
    plt.title(f"Top 20 Predictive CpG Sites ({MODEL_TYPE})")
    plt.xlabel("Coefficient Weight")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    filename = f"{MODEL_TYPE}_h{USE_HORVATH}_k{FEATURE_K}_cpg.png"
    plt.savefig(os.path.join(plot_dir, filename))
    plt.show()